In [23]:
import pandas as pd
import numpy as np
import os
from pymongo import MongoClient
import joblib
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA

In [24]:
MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    raise ValueError("configure a variavel MONGO_URI antes de rodar o notebook")
client = MongoClient(MONGO_URI)
db = client["PetMatch"]
colecao = db["pets"]

dados = list(colecao.find())
df = pd.DataFrame(dados)
df = df.drop(columns=["_id"], errors="ignore")

In [25]:
df = df[[
    "tipo_animal", "sexo", "porte", "idade", "pelagem",
    "cuidados_veterinarios", "vive_bem_com", "sociavel_com"
]]

df = df.dropna()
df = df[df["sexo"].str.lower().str.strip() != "ambos"]

for col in df.columns:
    df[col] = df[col].astype(str).str.lower().str.strip()

In [26]:
df["sexo"] = df["sexo"].map({"fêmea": 0.0, "macho": 1.0})
df["porte"] = df["porte"].map({"pequeno": 0.0, "médio": 1.0, "grande": 2.0})
df["idade"] = df["idade"].map({
    "abaixo de 2 meses": 0.0, "2 a 6 meses": 1.0, "7 a 11 meses": 2.0,
    "1 ano": 3.0, "2 anos": 4.0, "3 anos": 5.0,
    "4 anos": 6.0, "5 anos": 7.0, "6 ou mais anos": 8.0
})

df = df.dropna(subset=["sexo", "porte", "idade"])

df_cachorro = df[df["tipo_animal"] == "cachorro"].drop(columns=["tipo_animal"]).reset_index(drop=True)
df_gato = df[df["tipo_animal"] == "gato"].drop(columns=["tipo_animal"]).reset_index(drop=True)

In [27]:
df_pelagem_c = df_cachorro["pelagem"].str.get_dummies(sep=", ")
df_cuidados_c = df_cachorro["cuidados_veterinarios"].str.get_dummies(sep=", ")
df_vive_c = df_cachorro["vive_bem_com"].str.get_dummies(sep=", ")
df_sociavel_c = df_cachorro["sociavel_com"].str.get_dummies(sep=", ")

df_pelagem_c.columns = ["pelagem_" + col for col in df_pelagem_c.columns]
df_cuidados_c.columns = ["cuidados_" + col for col in df_cuidados_c.columns]
df_vive_c.columns = ["vive_" + col for col in df_vive_c.columns]
df_sociavel_c.columns = ["sociavel_" + col for col in df_sociavel_c.columns]

if "cuidados_rajada" in df_cuidados_c.columns:
    df_cuidados_c = df_cuidados_c.drop(columns=["cuidados_rajada"])

df_cachorro_raw = pd.concat([
    df_cachorro[["sexo", "porte", "idade"]],
    df_pelagem_c, df_cuidados_c, df_vive_c, df_sociavel_c
], axis=1).dropna()

In [28]:
df_pelagem_g = df_gato["pelagem"].str.get_dummies(sep=", ")
df_cuidados_g = df_gato["cuidados_veterinarios"].str.get_dummies(sep=", ")
df_vive_g = df_gato["vive_bem_com"].str.get_dummies(sep=", ")
df_sociavel_g = df_gato["sociavel_com"].str.get_dummies(sep=", ")

df_pelagem_g.columns = ["pelagem_" + col for col in df_pelagem_g.columns]
df_cuidados_g.columns = ["cuidados_" + col for col in df_cuidados_g.columns]
df_vive_g.columns = ["vive_" + col for col in df_vive_g.columns]
df_sociavel_g.columns = ["sociavel_" + col for col in df_sociavel_g.columns]

if "cuidados_rajada" in df_cuidados_g.columns:
    df_cuidados_g = df_cuidados_g.drop(columns=["cuidados_rajada"])

df_gato_raw = pd.concat([
    df_gato[["sexo", "porte", "idade"]],
    df_pelagem_g, df_cuidados_g, df_vive_g, df_sociavel_g
], axis=1).dropna()

In [29]:
scaler_cachorro = MinMaxScaler()
df_cachorro_norm = pd.DataFrame(
    scaler_cachorro.fit_transform(df_cachorro_raw),
    columns=df_cachorro_raw.columns
)

pca_cachorro = PCA(n_components=0.9)
dados_cachorro = pca_cachorro.fit_transform(df_cachorro_norm)

print(f"CACHORRO: {df_cachorro_raw.shape[1]} → {dados_cachorro.shape[1]}")

CACHORRO: 46 → 16


In [30]:
scaler_gato = MinMaxScaler()
df_gato_norm = pd.DataFrame(
    scaler_gato.fit_transform(df_gato_raw),
    columns=df_gato_raw.columns
)

pca_gato = PCA(n_components=0.9)
dados_gato = pca_gato.fit_transform(df_gato_norm)

print(f"GATO: {df_gato_raw.shape[1]} → {dados_gato.shape[1]}")

GATO: 50 → 16


In [31]:
from sklearn.metrics.pairwise import euclidean_distances

dist_cachorro = euclidean_distances(dados_cachorro)
dist_gato = euclidean_distances(dados_gato)

In [32]:
MODEL_DIR = Path("backend/fastapi_app/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler_cachorro, MODEL_DIR / "scaler_cachorro.pkl")
joblib.dump(scaler_gato, MODEL_DIR / "scaler_gato.pkl")
joblib.dump(pca_cachorro, MODEL_DIR / "pca_cachorro.pkl")
joblib.dump(pca_gato, MODEL_DIR / "pca_gato.pkl")
joblib.dump(df_cachorro_raw.columns.tolist(), MODEL_DIR / "colunas_cachorro.pkl")
joblib.dump(df_gato_raw.columns.tolist(), MODEL_DIR / "colunas_gato.pkl")

['backend\\fastapi_app\\models\\colunas_gato.pkl']

In [33]:
client = MongoClient(MONGO_URI)
db = client["PetMatch"]
colecao = db["pets"]

dados_mongo = list(colecao.find())
df_mongo = pd.DataFrame(dados_mongo)

idx_cachorro = df_mongo[df_mongo["tipo_animal"] == "cachorro"].index.tolist()
idx_gato = df_mongo[df_mongo["tipo_animal"] == "gato"].index.tolist()

joblib.dump(idx_cachorro, MODEL_DIR / "idx_cachorro.pkl")
joblib.dump(idx_gato, MODEL_DIR / "idx_gato.pkl")

['backend\\fastapi_app\\models\\idx_gato.pkl']